In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for file in files[:10]:
        print("   ", file)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/moomnataskeen
/kaggle/input/datasets/moomnataskeen/football-data-co-uk
    E0 (1).csv
    E0 (10).csv
    E0 (11).csv
    E0 (12).csv
    E0 (13).csv
    E0 (14).csv
    E0 (15).csv
    E0 (16).csv
    E0 (17).csv
    E0 (18).csv


In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    GradientBoostingClassifier,
    StackingClassifier
)

from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve
)

from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss
)

import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

In [4]:

import os

for root, dirs, files in os.walk("/kaggle/input/football-data-co-uk"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))



In [5]:

import os

print("Datasets available in Kaggle:")

print(os.listdir("/kaggle/input"))


Datasets available in Kaggle:
['datasets']


In [6]:

import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print("Folder:", root)
    print("Files:", files)


Folder: /kaggle/input/datasets
Files: []
Folder: /kaggle/input/datasets/moomnataskeen
Files: []
Folder: /kaggle/input/datasets/moomnataskeen/football-data-co-uk
Files: ['E0 (1).csv', 'E0 (10).csv', 'E0 (11).csv', 'E0 (12).csv', 'E0 (13).csv', 'E0 (14).csv', 'E0 (15).csv', 'E0 (16).csv', 'E0 (17).csv', 'E0 (18).csv', 'E0 (19).csv', 'E0 (2).csv', 'E0 (20).csv', 'E0 (3).csv', 'E0 (4).csv', 'E0 (5).csv', 'E0 (6).csv', 'E0 (7).csv', 'E0 (8).csv', 'E0 (9).csv', 'E0.csv']


In [7]:

import pandas as pd
import glob
import os

# Find all CSV files in your football dataset folder
folder_path = "/kaggle/input/datasets/moomnataskeen/football-data-co-uk"
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Load each CSV file
dataframes = []

for file in csv_files:
    temp_df = pd.read_csv(file, encoding="latin1")
    dataframes.append(temp_df)

# Combine all seasons into one dataset
df = pd.concat(dataframes, ignore_index=True)

print("Total CSV files loaded:", len(csv_files))
print("Total matches/rows:", df.shape[0])
print("Total columns:", df.shape[1])

# Display the first five rows
df.head()


Total CSV files loaded: 21
Total matches/rows: 7671
Total columns: 207


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,PPA,SKBH,SKBD,SKBA,PPCH,PPCD,PPCA,SKBCH,SKBCD,SKBCA
0,E0,11/08/07,Aston Villa,Liverpool,1.0,2.0,A,0.0,1.0,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,11/08/07,Bolton,Newcastle,1.0,3.0,A,0.0,3.0,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,11/08/07,Derby,Portsmouth,2.0,2.0,D,1.0,1.0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,11/08/07,Everton,Wigan,2.0,1.0,H,1.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,11/08/07,Middlesbrough,Blackburn,1.0,2.0,A,1.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:

# Check whether our required columns exist
required_columns = [
    "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR"
]

print("Required columns available:")
for col in required_columns:
    print(col, ":", col in df.columns)

# Check the date range
print("\nOriginal date values:")
print(df["Date"].head(10))

# Check the match result categories
print("\nMatch result categories:")
print(df["FTR"].value_counts(dropna=False))


Required columns available:
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True

Original date values:
0    11/08/07
1    11/08/07
2    11/08/07
3    11/08/07
4    11/08/07
5    11/08/07
6    11/08/07
7    12/08/07
8    12/08/07
9    12/08/07
Name: Date, dtype: object

Match result categories:
FTR
H      3472
A      2314
D      1884
NaN       1
Name: count, dtype: int64


In [9]:

# Check whether our required columns exist
required_columns = [
    "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR"
]

print("Required columns available:")
for col in required_columns:
    print(col, ":", col in df.columns)

# Check the date range
print("\nOriginal date values:")
print(df["Date"].head(10))

# Check the match result categories
print("\nMatch result categories:")
print(df["FTR"].value_counts(dropna=False))


Required columns available:
Date : True
HomeTeam : True
AwayTeam : True
FTHG : True
FTAG : True
FTR : True

Original date values:
0    11/08/07
1    11/08/07
2    11/08/07
3    11/08/07
4    11/08/07
5    11/08/07
6    11/08/07
7    12/08/07
8    12/08/07
9    12/08/07
Name: Date, dtype: object

Match result categories:
FTR
H      3472
A      2314
D      1884
NaN       1
Name: count, dtype: int64


In [10]:

df = df.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
).copy()

df["Date"] = pd.to_datetime(
    df["Date"], dayfirst=True, errors="coerce"
)

df["FTHG"] = pd.to_numeric(df["FTHG"], errors="coerce")
df["FTAG"] = pd.to_numeric(df["FTAG"], errors="coerce")

df = df.dropna(
    subset=["Date", "FTHG", "FTAG"]
).copy()

df = df[df["FTR"].isin(["H", "D", "A"])].copy()

df = df.sort_values("Date").reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("\nResult distribution:")
print(df["FTR"].value_counts())


Rows after cleaning: 7670
Date range: 2006-08-19 00:00:00 to 2026-10-19 00:00:00

Result distribution:
FTR
H    3472
A    2314
D    1884
Name: count, dtype: int64


In [11]:

# Home team's perspective
home_history = df[["Date", "HomeTeam", "FTHG", "FTAG", "FTR"]].copy()

home_history.columns = [
    "Date", "Team", "GoalsFor", "GoalsAgainst", "Result"
]

home_history["Points"] = home_history["Result"].map({
    "H": 3,  # Home team won
    "D": 1,  # Draw
    "A": 0   # Home team lost
})

# Away team's perspective
away_history = df[["Date", "AwayTeam", "FTAG", "FTHG", "FTR"]].copy()

away_history.columns = [
    "Date", "Team", "GoalsFor", "GoalsAgainst", "Result"
]

away_history["Points"] = away_history["Result"].map({
    "A": 3,  # Away team won
    "D": 1,  # Draw
    "H": 0   # Away team lost
})

# Combine both perspectives
team_history = pd.concat(
    [home_history, away_history],
    ignore_index=True
)

team_history = team_history.sort_values(
    ["Team", "Date"]
).reset_index(drop=True)

print("Team-history rows:", len(team_history))
print(team_history.head(10))


Team-history rows: 15340
        Date     Team  GoalsFor  GoalsAgainst Result  Points
0 2006-08-19  Arsenal       1.0           1.0      D       1
1 2006-08-26  Arsenal       0.0           1.0      H       0
2 2006-09-09  Arsenal       1.0           1.0      D       1
3 2006-09-17  Arsenal       1.0           0.0      A       3
4 2006-09-23  Arsenal       3.0           0.0      H       3
5 2006-09-30  Arsenal       2.0           1.0      A       3
6 2006-10-14  Arsenal       3.0           0.0      H       3
7 2006-10-22  Arsenal       4.0           0.0      A       3
8 2006-10-28  Arsenal       1.0           1.0      D       1
9 2006-11-05  Arsenal       0.0           1.0      H       0


In [12]:

# Calculate rolling form separately for each team
team_history["FormPoints5"] = (
    team_history.groupby("Team")["Points"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

team_history["AvgGoalsFor5"] = (
    team_history.groupby("Team")["GoalsFor"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

team_history["AvgGoalsAgainst5"] = (
    team_history.groupby("Team")["GoalsAgainst"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

print(
    team_history[
        [
            "Date", "Team", "GoalsFor", "GoalsAgainst",
            "Points", "FormPoints5", "AvgGoalsFor5",
            "AvgGoalsAgainst5"
        ]
    ].head(15)
)


         Date     Team  GoalsFor  GoalsAgainst  Points  FormPoints5  \
0  2006-08-19  Arsenal       1.0           1.0       1          NaN   
1  2006-08-26  Arsenal       0.0           1.0       0     1.000000   
2  2006-09-09  Arsenal       1.0           1.0       1     0.500000   
3  2006-09-17  Arsenal       1.0           0.0       3     0.666667   
4  2006-09-23  Arsenal       3.0           0.0       3     1.250000   
5  2006-09-30  Arsenal       2.0           1.0       3     1.600000   
6  2006-10-14  Arsenal       3.0           0.0       3     2.000000   
7  2006-10-22  Arsenal       4.0           0.0       3     2.600000   
8  2006-10-28  Arsenal       1.0           1.0       1     3.000000   
9  2006-11-05  Arsenal       0.0           1.0       0     2.600000   
10 2006-11-12  Arsenal       3.0           0.0       3     2.000000   
11 2006-11-18  Arsenal       1.0           1.0       1     2.000000   
12 2006-11-25  Arsenal       1.0           3.0       0     1.600000   
13 200

In [13]:

# Make sure matches are ordered by date
df = df.sort_values("Date").reset_index(drop=True)

# Create home and away team-history records
home_history = df[
    ["Date", "HomeTeam", "FTHG", "FTAG", "FTR"]
].copy()

home_history.columns = [
    "Date", "Team", "GoalsFor", "GoalsAgainst", "Result"
]

home_history["Points"] = home_history["Result"].map({
    "H": 3, "D": 1, "A": 0
})

away_history = df[
    ["Date", "AwayTeam", "FTAG", "FTHG", "FTR"]
].copy()

away_history.columns = [
    "Date", "Team", "GoalsFor", "GoalsAgainst", "Result"
]

away_history["Points"] = away_history["Result"].map({
    "A": 3, "D": 1, "H": 0
})

team_history = pd.concat(
    [home_history, away_history],
    ignore_index=True
)

# Combine all matches played by a team on the same date first.
# Then calculate form using only dates before the current date.
daily_history = (
    team_history
    .groupby(["Team", "Date"], as_index=False)
    .agg(
        GoalsFor=("GoalsFor", "mean"),
        GoalsAgainst=("GoalsAgainst", "mean"),
        Points=("Points", "mean")
    )
    .sort_values(["Team", "Date"])
    .reset_index(drop=True)
)

def calculate_form(group):
    group = group.sort_values("Date").copy()

    group["FormPoints5"] = (
        group["Points"].shift(1).rolling(5, min_periods=1).mean()
    )

    group["AvgGoalsFor5"] = (
        group["GoalsFor"].shift(1).rolling(5, min_periods=1).mean()
    )

    group["AvgGoalsAgainst5"] = (
        group["GoalsAgainst"].shift(1).rolling(5, min_periods=1).mean()
    )

    return group

daily_history = (
    daily_history.groupby("Team", group_keys=False)
    .apply(calculate_form)
    .reset_index(drop=True)
)

print("Daily team-history rows:", len(daily_history))
print(daily_history.head(10))


Daily team-history rows: 15340
      Team       Date  GoalsFor  GoalsAgainst  Points  FormPoints5  \
0  Arsenal 2006-08-19       1.0           1.0     1.0          NaN   
1  Arsenal 2006-08-26       0.0           1.0     0.0     1.000000   
2  Arsenal 2006-09-09       1.0           1.0     1.0     0.500000   
3  Arsenal 2006-09-17       1.0           0.0     3.0     0.666667   
4  Arsenal 2006-09-23       3.0           0.0     3.0     1.250000   
5  Arsenal 2006-09-30       2.0           1.0     3.0     1.600000   
6  Arsenal 2006-10-14       3.0           0.0     3.0     2.000000   
7  Arsenal 2006-10-22       4.0           0.0     3.0     2.600000   
8  Arsenal 2006-10-28       1.0           1.0     1.0     3.000000   
9  Arsenal 2006-11-05       0.0           1.0     0.0     2.600000   

   AvgGoalsFor5  AvgGoalsAgainst5  
0           NaN               NaN  
1      1.000000              1.00  
2      0.500000              1.00  
3      0.666667              1.00  
4      0.750000   

In [14]:

# Give the home team's form columns distinct names
home_form = daily_history[
    ["Team", "Date", "FormPoints5", "AvgGoalsFor5", "AvgGoalsAgainst5"]
].copy()

home_form = home_form.rename(columns={
    "Team": "HomeTeam",
    "FormPoints5": "HomeFormPoints5",
    "AvgGoalsFor5": "HomeAvgGoalsFor5",
    "AvgGoalsAgainst5": "HomeAvgGoalsAgainst5"
})

# Give the away team's form columns distinct names
away_form = daily_history[
    ["Team", "Date", "FormPoints5", "AvgGoalsFor5", "AvgGoalsAgainst5"]
].copy()

away_form = away_form.rename(columns={
    "Team": "AwayTeam",
    "FormPoints5": "AwayFormPoints5",
    "AvgGoalsFor5": "AwayAvgGoalsFor5",
    "AvgGoalsAgainst5": "AwayAvgGoalsAgainst5"
})

# Attach home-team form to each match
df = df.merge(
    home_form,
    on=["Date", "HomeTeam"],
    how="left",
    validate="many_to_one"
)

# Attach away-team form to each match
df = df.merge(
    away_form,
    on=["Date", "AwayTeam"],
    how="left",
    validate="many_to_one"
)

print("Dataset shape:", df.shape)

print(
    df[
        [
            "Date", "HomeTeam", "AwayTeam", "FTR",
            "HomeFormPoints5", "AwayFormPoints5",
            "HomeAvgGoalsFor5", "AwayAvgGoalsFor5"
        ]
    ].head(10)
)


Dataset shape: (7670, 213)
        Date          HomeTeam       AwayTeam FTR  HomeFormPoints5  \
0 2006-08-19           Arsenal    Aston Villa   D              NaN   
1 2006-08-19            Bolton      Tottenham   H              NaN   
2 2006-08-19           Everton        Watford   H              NaN   
3 2006-08-19         Newcastle          Wigan   H              NaN   
4 2006-08-19        Portsmouth      Blackburn   H              NaN   
5 2006-08-19           Reading  Middlesbrough   H              NaN   
6 2006-08-19  Sheffield United      Liverpool   D              NaN   
7 2006-08-19          West Ham       Charlton   H              NaN   
8 2006-08-20        Man United         Fulham   H              NaN   
9 2006-08-20           Chelsea       Man City   H              NaN   

   AwayFormPoints5  HomeAvgGoalsFor5  AwayAvgGoalsFor5  
0              NaN               NaN               NaN  
1              NaN               NaN               NaN  
2              NaN             

In [15]:

form_columns = [
    "HomeFormPoints5",
    "AwayFormPoints5",
    "HomeAvgGoalsFor5",
    "AwayAvgGoalsFor5",
    "HomeAvgGoalsAgainst5",
    "AwayAvgGoalsAgainst5"
]

print("Missing values in form features:")
print(df[form_columns].isnull().sum())

print("\nExample match with form features:")
print(df[[
    "Date", "HomeTeam", "AwayTeam", "FTR"
] + form_columns].iloc[20])


Missing values in form features:
HomeFormPoints5         21
AwayFormPoints5         24
HomeAvgGoalsFor5        21
AwayAvgGoalsFor5        24
HomeAvgGoalsAgainst5    21
AwayAvgGoalsAgainst5    24
dtype: int64

Example match with form features:
Date                    2006-08-26 00:00:00
HomeTeam                            Watford
AwayTeam                         Man United
FTR                                       A
HomeFormPoints5                         0.5
AwayFormPoints5                         3.0
HomeAvgGoalsFor5                        1.0
AwayAvgGoalsFor5                        4.0
HomeAvgGoalsAgainst5                    1.5
AwayAvgGoalsAgainst5                    0.5
Name: 20, dtype: object


In [16]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.dummy import DummyClassifier

feature_columns = [
    "HomeFormPoints5",
    "AwayFormPoints5",
    "HomeAvgGoalsFor5",
    "AwayAvgGoalsFor5",
    "HomeAvgGoalsAgainst5",
    "AwayAvgGoalsAgainst5"
]

X = df[feature_columns].copy()
y = df["FTR"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget classes:")
print(y.value_counts())


Feature matrix shape: (7670, 6)
Target shape: (7670,)

Target classes:
FTR
H    3472
A    2314
D    1884
Name: count, dtype: int64


In [17]:

# Sort by date to ensure chronological order
df = df.sort_values("Date").reset_index(drop=True)

X = df[feature_columns].copy()
y = df["FTR"].copy()

# Use the first 80% of matches for training
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training matches:", len(X_train))
print("Testing matches:", len(X_test))
print("Last training date:", df["Date"].iloc[split_index - 1])
print("First testing date:", df["Date"].iloc[split_index])


Training matches: 6136
Testing matches: 1534
Last training date: 2022-09-03 00:00:00
First testing date: 2022-09-03 00:00:00


In [18]:

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test, baseline_predictions
)

print("Baseline accuracy:", round(baseline_accuracy, 4))


Baseline accuracy: 0.4413


In [19]:

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

model_accuracy = accuracy_score(y_test, predictions)

print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Logistic Regression accuracy:", round(model_accuracy, 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))


Baseline accuracy: 0.4413
Logistic Regression accuracy: 0.5013

Classification report:
              precision    recall  f1-score   support

    Home Win       0.50      0.85      0.63       677
        Draw       0.00      0.00      0.00       369
    Away Win       0.50      0.40      0.45       488

    accuracy                           0.50      1534
   macro avg       0.33      0.42      0.36      1534
weighted avg       0.38      0.50      0.42      1534



In [20]:

from sklearn.metrics import f1_score, confusion_matrix

# Create a new model that gives extra weight to less frequent classes
balanced_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

# Train on the same training data
balanced_model.fit(X_train, y_train)

# Predict the test matches
balanced_predictions = balanced_model.predict(X_test)

# Evaluate the new model
balanced_accuracy = accuracy_score(
    y_test, balanced_predictions
)

balanced_macro_f1 = f1_score(
    y_test, balanced_predictions,
    labels=["H", "D", "A"],
    average="macro",
    zero_division=0
)

print("Original model accuracy:", round(model_accuracy, 4))
print("Balanced model accuracy:", round(balanced_accuracy, 4))
print("Balanced model macro F1:", round(balanced_macro_f1, 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    balanced_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))

print("Confusion matrix (rows = actual, columns = predicted)")
print("Order: Home Win, Draw, Away Win")
print(confusion_matrix(
    y_test,
    balanced_predictions,
    labels=["H", "D", "A"]
))


Original model accuracy: 0.5013
Balanced model accuracy: 0.4557
Balanced model macro F1: 0.4061

Classification report:
              precision    recall  f1-score   support

    Home Win       0.54      0.55      0.55       677
        Draw       0.22      0.14      0.17       369
    Away Win       0.45      0.57      0.50       488

    accuracy                           0.46      1534
   macro avg       0.40      0.42      0.41      1534
weighted avg       0.44      0.46      0.44      1534

Confusion matrix (rows = actual, columns = predicted)
Order: Home Win, Draw, Away Win
[[370 111 196]
 [173  52 144]
 [136  75 277]]


In [21]:

# History of matches played at home
home_venue = df[
    ["Date", "HomeTeam", "FTHG", "FTAG", "FTR"]
].copy()

home_venue.columns = [
    "Date", "Team", "GoalsFor", "GoalsAgainst", "Result"
]

home_venue["Points"] = home_venue["Result"].map({
    "H": 3, "D": 1, "A": 0
})

# One record per team per date
home_venue = (
    home_venue.groupby(["Team", "Date"], as_index=False)
    .agg(
        GoalsFor=("GoalsFor", "mean"),
        GoalsAgainst=("GoalsAgainst", "mean"),
        Points=("Points", "mean")
    )
    .sort_values(["Team", "Date"])
)

def add_venue_form(group):
    group = group.sort_values("Date").copy()

    group["VenuePoints5"] = (
        group["Points"].shift(1).rolling(5, min_periods=1).mean()
    )
    group["VenueGoalsFor5"] = (
        group["GoalsFor"].shift(1).rolling(5, min_periods=1).mean()
    )
    group["VenueGoalsAgainst5"] = (
        group["GoalsAgainst"].shift(1).rolling(5, min_periods=1).mean()
    )

    return group

home_venue = (
    home_venue.groupby("Team", group_keys=False)
    .apply(add_venue_form)
    .reset_index(drop=True)
)

print(home_venue.head(10))


      Team       Date  GoalsFor  GoalsAgainst  Points  VenuePoints5  \
0  Arsenal 2006-08-19       1.0           1.0     1.0           NaN   
1  Arsenal 2006-09-09       1.0           1.0     1.0      1.000000   
2  Arsenal 2006-09-23       3.0           0.0     3.0      1.000000   
3  Arsenal 2006-10-14       3.0           0.0     3.0      1.666667   
4  Arsenal 2006-10-28       1.0           1.0     1.0      2.000000   
5  Arsenal 2006-11-12       3.0           0.0     3.0      1.800000   
6  Arsenal 2006-11-18       1.0           1.0     1.0      2.200000   
7  Arsenal 2006-12-02       3.0           0.0     3.0      2.200000   
8  Arsenal 2006-12-16       2.0           2.0     1.0      2.200000   
9  Arsenal 2006-12-23       6.0           2.0     3.0      1.800000   

   VenueGoalsFor5  VenueGoalsAgainst5  
0             NaN                 NaN  
1        1.000000            1.000000  
2        1.000000            1.000000  
3        1.666667            0.666667  
4        2.000000 

In [22]:

# History of matches played away
away_venue = df[
    ["Date", "AwayTeam", "FTAG", "FTHG", "FTR"]
].copy()

away_venue.columns = [
    "Date", "Team", "GoalsFor", "GoalsAgainst", "Result"
]

away_venue["Points"] = away_venue["Result"].map({
    "A": 3, "D": 1, "H": 0
})

away_venue = (
    away_venue.groupby(["Team", "Date"], as_index=False)
    .agg(
        GoalsFor=("GoalsFor", "mean"),
        GoalsAgainst=("GoalsAgainst", "mean"),
        Points=("Points", "mean")
    )
    .sort_values(["Team", "Date"])
)

away_venue = (
    away_venue.groupby("Team", group_keys=False)
    .apply(add_venue_form)
    .reset_index(drop=True)
)

print(away_venue.head(10))


      Team       Date  GoalsFor  GoalsAgainst  Points  VenuePoints5  \
0  Arsenal 2006-08-26       0.0           1.0     0.0           NaN   
1  Arsenal 2006-09-17       1.0           0.0     3.0          0.00   
2  Arsenal 2006-09-30       2.0           1.0     3.0          1.50   
3  Arsenal 2006-10-22       4.0           0.0     3.0          2.00   
4  Arsenal 2006-11-05       0.0           1.0     0.0          2.25   
5  Arsenal 2006-11-25       1.0           3.0     0.0          1.80   
6  Arsenal 2006-11-29       1.0           2.0     0.0          1.80   
7  Arsenal 2006-12-10       1.0           1.0     1.0          1.20   
8  Arsenal 2006-12-13       1.0           0.0     3.0          0.80   
9  Arsenal 2006-12-26       2.0           1.0     3.0          0.80   

   VenueGoalsFor5  VenueGoalsAgainst5  
0             NaN                 NaN  
1            0.00            1.000000  
2            0.50            0.500000  
3            1.00            0.666667  
4            1.75 

In [23]:

# Select and rename home-specific features
home_features = home_venue[
    ["Date", "Team", "VenuePoints5",
     "VenueGoalsFor5", "VenueGoalsAgainst5"]
].rename(columns={
    "Team": "HomeTeam",
    "VenuePoints5": "HomeVenuePoints5",
    "VenueGoalsFor5": "HomeVenueGoalsFor5",
    "VenueGoalsAgainst5": "HomeVenueGoalsAgainst5"
})

# Select and rename away-specific features
away_features = away_venue[
    ["Date", "Team", "VenuePoints5",
     "VenueGoalsFor5", "VenueGoalsAgainst5"]
].rename(columns={
    "Team": "AwayTeam",
    "VenuePoints5": "AwayVenuePoints5",
    "VenueGoalsFor5": "AwayVenueGoalsFor5",
    "VenueGoalsAgainst5": "AwayVenueGoalsAgainst5"
})

# Remove old copies if this cell is run more than once
new_features = [
    "HomeVenuePoints5", "HomeVenueGoalsFor5",
    "HomeVenueGoalsAgainst5", "AwayVenuePoints5",
    "AwayVenueGoalsFor5", "AwayVenueGoalsAgainst5"
]

df = df.drop(columns=new_features, errors="ignore")

# Merge home and away venue form
df = df.merge(
    home_features,
    on=["Date", "HomeTeam"],
    how="left",
    validate="many_to_one"
)

df = df.merge(
    away_features,
    on=["Date", "AwayTeam"],
    how="left",
    validate="many_to_one"
)

print("Updated dataset shape:", df.shape)
print(df[["HomeTeam", "AwayTeam"] + new_features].head(10))


Updated dataset shape: (7670, 219)
           HomeTeam       AwayTeam  HomeVenuePoints5  HomeVenueGoalsFor5  \
0           Arsenal    Aston Villa               NaN                 NaN   
1            Bolton      Tottenham               NaN                 NaN   
2           Everton        Watford               NaN                 NaN   
3         Newcastle          Wigan               NaN                 NaN   
4        Portsmouth      Blackburn               NaN                 NaN   
5           Reading  Middlesbrough               NaN                 NaN   
6  Sheffield United      Liverpool               NaN                 NaN   
7          West Ham       Charlton               NaN                 NaN   
8        Man United         Fulham               NaN                 NaN   
9           Chelsea       Man City               NaN                 NaN   

   HomeVenueGoalsAgainst5  AwayVenuePoints5  AwayVenueGoalsFor5  \
0                     NaN               NaN                 N

In [27]:

venue_columns = [
    "HomeVenuePoints5",
    "HomeVenueGoalsFor5",
    "HomeVenueGoalsAgainst5",
    "AwayVenuePoints5",
    "AwayVenueGoalsFor5",
    "AwayVenueGoalsAgainst5"
]

all_features = feature_columns + venue_columns

df = df.sort_values("Date").reset_index(drop=True)

split_index = int(len(df) * 0.80)

X_train_new = df[all_features].iloc[:split_index]
X_test_new = df[all_features].iloc[split_index:]

y_train_new = df["FTR"].iloc[:split_index]
y_test_new = df["FTR"].iloc[split_index:]

improved_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

improved_model.fit(X_train_new, y_train_new)

new_predictions = improved_model.predict(X_test_new)

print("Original model accuracy:", round(model_accuracy, 4))
print(
    "Improved model accuracy:",
    round(accuracy_score(y_test_new, new_predictions), 4)
)

print(
    "Improved macro F1:",
    round(f1_score(
        y_test_new, new_predictions,
        labels=["H", "D", "A"],
        average="macro",
        zero_division=0
    ), 4)
)

print("\nClassification report:")
print(classification_report(
    y_test_new,
    new_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))


Original model accuracy: 0.5013
Improved model accuracy: 0.513
Improved macro F1: 0.3736

Classification report:
              precision    recall  f1-score   support

    Home Win       0.52      0.83      0.64       677
        Draw       0.00      0.00      0.00       369
    Away Win       0.51      0.47      0.49       488

    accuracy                           0.51      1534
   macro avg       0.34      0.43      0.37      1534
weighted avg       0.39      0.51      0.43      1534



In [28]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

balanced_improved_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

balanced_improved_model.fit(X_train_new, y_train_new)

balanced_new_predictions = balanced_improved_model.predict(X_test_new)

print("Accuracy:",
      round(accuracy_score(y_test_new, balanced_new_predictions), 4))

print("Macro F1:",
      round(f1_score(
          y_test_new,
          balanced_new_predictions,
          labels=["H", "D", "A"],
          average="macro",
          zero_division=0
      ), 4))

print("\nClassification Report:")
print(classification_report(
    y_test_new,
    balanced_new_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_test_new,
    balanced_new_predictions,
    labels=["H", "D", "A"]
))


Accuracy: 0.4459
Macro F1: 0.4055

Classification Report:
              precision    recall  f1-score   support

    Home Win       0.55      0.52      0.53       677
        Draw       0.23      0.17      0.20       369
    Away Win       0.43      0.56      0.49       488

    accuracy                           0.45      1534
   macro avg       0.41      0.41      0.41      1534
weighted avg       0.44      0.45      0.44      1534

Confusion Matrix:
[[349 121 207]
 [154  63 152]
 [126  90 272]]


In [29]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

random_forest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
)

random_forest_model.fit(X_train_new, y_train_new)

rf_predictions = random_forest_model.predict(X_test_new)

rf_accuracy = accuracy_score(y_test_new, rf_predictions)

rf_macro_f1 = f1_score(
    y_test_new,
    rf_predictions,
    labels=["H", "D", "A"],
    average="macro",
    zero_division=0
)

print("Random Forest Accuracy:", round(rf_accuracy, 4))
print("Random Forest Macro F1:", round(rf_macro_f1, 4))

print("\nClassification Report:")
print(classification_report(
    y_test_new,
    rf_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))


Random Forest Accuracy: 0.4452
Random Forest Macro F1: 0.4133

Classification Report:
              precision    recall  f1-score   support

    Home Win       0.54      0.52      0.53       677
        Draw       0.24      0.21      0.23       369
    Away Win       0.45      0.52      0.48       488

    accuracy                           0.45      1534
   macro avg       0.41      0.42      0.41      1534
weighted avg       0.44      0.45      0.44      1534



In [30]:

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, accuracy_score, f1_score

models = {
    "Regular Logistic Regression": improved_model,
    "Balanced Logistic Regression": balanced_improved_model,
    "Random Forest": random_forest_model
}

# Keep the same outcome order for every model.
outcome_order = ["H", "D", "A"]

results = []

for name, fitted_model in models.items():

    # Predict probabilities for the test matches.
    probabilities = fitted_model.predict_proba(X_test_new)

    # Reorder probability columns to H, D, A.
    model_classes = list(fitted_model.classes_)
    column_order = [model_classes.index(c) for c in outcome_order]
    probabilities = probabilities[:, column_order]

    predictions = fitted_model.predict(X_test_new)

    # Convert actual results into H, D, A numeric positions.
    actual_positions = np.array([
        outcome_order.index(result)
        for result in y_test_new
    ])

    # Create one-hot actual outcomes.
    actual_one_hot = np.eye(3)[actual_positions]

    # Calculate multiclass Brier score.
    brier = np.mean(
        np.sum((probabilities - actual_one_hot) ** 2, axis=1)
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_new, predictions),
        "Macro F1": f1_score(
            y_test_new,
            predictions,
            labels=outcome_order,
            average="macro",
            zero_division=0
        ),
        "Log Loss": log_loss(
            y_test_new,
            probabilities,
            labels=outcome_order
        ),
        "Brier Score": brier
    })

comparison = pd.DataFrame(results).round(4)

print(comparison.to_string(index=False))


                       Model  Accuracy  Macro F1  Log Loss  Brier Score
 Regular Logistic Regression    0.5130    0.3736    1.2819       0.6121
Balanced Logistic Regression    0.4459    0.4055    1.2632       0.6332
               Random Forest    0.4452    0.4133    1.2481       0.6328


In [31]:

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

outcomes = ["H", "D", "A"]

# Baseline 1: predict equal probability for every outcome.
uniform_probs = np.full((len(y_test_new), 3), 1/3)

# Baseline 2: use outcome frequencies from training data only.
train_frequencies = (
    y_train_new.value_counts(normalize=True)
    .reindex(outcomes, fill_value=0)
    .values
)

prior_probs = np.tile(
    train_frequencies,
    (len(y_test_new), 1)
)

actual = np.eye(3)[
    [outcomes.index(value) for value in y_test_new]
]

def brier_score(probs, actual):
    return np.mean(np.sum((probs - actual) ** 2, axis=1))

print("Uniform baseline")
print("Log loss:", round(log_loss(
    y_test_new, uniform_probs, labels=outcomes
), 4))
print("Brier score:", round(brier_score(
    uniform_probs, actual
), 4))

print("\nTraining-frequency baseline")
print("Log loss:", round(log_loss(
    y_test_new, prior_probs, labels=outcomes
), 4))
print("Brier score:", round(brier_score(
    prior_probs, actual
), 4))


Uniform baseline
Log loss: 1.0986
Brier score: 0.6667

Training-frequency baseline
Log loss: 1.1215
Brier score: 0.6468


In [32]:

from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score, f1_score
import numpy as np

base_rf = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
)

calibrated_rf = CalibratedClassifierCV(
    estimator=base_rf,
    method="sigmoid",
    cv=TimeSeriesSplit(n_splits=3)
)

calibrated_rf.fit(X_train_new, y_train_new)

calibrated_probs_raw = calibrated_rf.predict_proba(X_test_new)

# Ensure columns are in H, D, A order.
class_order = list(calibrated_rf.classes_)
column_order = [class_order.index(c) for c in ["H", "D", "A"]]
calibrated_probs = calibrated_probs_raw[:, column_order]

calibrated_predictions = np.array(["H", "D", "A"])[
    np.argmax(calibrated_probs, axis=1)
]

actual_positions = np.array([
    ["H", "D", "A"].index(result)
    for result in y_test_new
])
actual_one_hot = np.eye(3)[actual_positions]

calibrated_brier = np.mean(
    np.sum((calibrated_probs - actual_one_hot) ** 2, axis=1)
)

print("Calibrated Random Forest")
print("Accuracy:", round(
    accuracy_score(y_test_new, calibrated_predictions), 4
))
print("Macro F1:", round(
    f1_score(
        y_test_new,
        calibrated_predictions,
        labels=["H", "D", "A"],
        average="macro",
        zero_division=0
    ), 4
))
print("Log loss:", round(
    log_loss(y_test_new, calibrated_probs, labels=["H", "D", "A"]), 4
))
print("Brier score:", round(calibrated_brier, 4))


Calibrated Random Forest
Accuracy: 0.498
Macro F1: 0.3586
Log loss: 1.2238
Brier score: 0.6164


In [33]:

from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(
    y_test_new,
    calibrated_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_test_new,
    calibrated_predictions,
    labels=["H", "D", "A"]
))

print("\nExample probabilities (first 10 matches):")

probability_examples = pd.DataFrame({
    "Actual": y_test_new.iloc[:10].values,
    "Predicted": calibrated_predictions[:10],
    "Home Win Probability": calibrated_probs[:10, 0].round(3),
    "Draw Probability": calibrated_probs[:10, 1].round(3),
    "Away Win Probability": calibrated_probs[:10, 2].round(3)
})

print(probability_examples.to_string(index=False))


Classification Report:
              precision    recall  f1-score   support

    Home Win       0.50      0.82      0.63       677
        Draw       0.00      0.00      0.00       369
    Away Win       0.48      0.42      0.45       488

    accuracy                           0.50      1534
   macro avg       0.33      0.42      0.36      1534
weighted avg       0.38      0.50      0.42      1534

Confusion Matrix:
[[558   0 119]
 [266   0 103]
 [282   0 206]]

Example probabilities (first 10 matches):
Actual Predicted  Home Win Probability  Draw Probability  Away Win Probability
     D         H                 0.510             0.250                 0.239
     H         H                 0.446             0.266                 0.288
     H         H                 0.588             0.242                 0.171
     H         A                 0.296             0.265                 0.439
     H         H                 0.477             0.277                 0.246
     A         

In [34]:

import numpy as np

df = df.sort_values("Date").reset_index(drop=True)

ratings = {}
home_elo_values = np.zeros(len(df))
away_elo_values = np.zeros(len(df))

initial_rating = 1500
home_advantage = 60
k_factor = 20

for date, daily_matches in df.groupby("Date", sort=True):

    # First calculate every prediction using ratings
    # from before any match on this date is processed.
    updates = []

    for idx in daily_matches.index:
        home = df.at[idx, "HomeTeam"]
        away = df.at[idx, "AwayTeam"]

        home_rating = ratings.get(home, initial_rating)
        away_rating = ratings.get(away, initial_rating)

        home_elo_values[idx] = home_rating
        away_elo_values[idx] = away_rating

        # Home advantage is included in the expectation.
        expected_home = 1 / (
            1 + 10 ** (
                (away_rating - (home_rating + home_advantage)) / 400
            )
        )

        result = df.at[idx, "FTR"]

        if result == "H":
            actual_home = 1.0
        elif result == "D":
            actual_home = 0.5
        else:
            actual_home = 0.0

        updates.append(
            (home, away, home_rating, away_rating,
             expected_home, actual_home)
        )

    # Update ratings only after all matches for this date
    # have had their pre-match ratings recorded.
    for home, away, home_rating, away_rating, expected, actual in updates:
        change = k_factor * (actual - expected)

        ratings[home] = home_rating + change
        ratings[away] = away_rating - change

df["HomeElo"] = home_elo_values
df["AwayElo"] = away_elo_values
df["EloDifference"] = df["HomeElo"] - df["AwayElo"]

print(df[
    ["Date", "HomeTeam", "AwayTeam",
     "HomeElo", "AwayElo", "EloDifference", "FTR"]
].tail(10).to_string(index=False))


      Date      HomeTeam       AwayTeam     HomeElo     AwayElo  EloDifference FTR
2026-10-17       Everton        Chelsea 1588.320993 1621.205965     -32.884972   A
2026-10-17     Newcastle    Aston Villa 1591.122489 1638.857584     -47.735096   A
2026-10-17     Brentford      Liverpool 1620.756476 1683.123182     -62.366706   A
2026-10-17      Man City        Ipswich 1808.665238 1448.242553     360.422685   H
2026-10-17        Fulham           Hull 1546.115632 1467.951544      78.164088   H
2026-10-18   Bournemouth     Sunderland 1616.554315 1527.151580      89.402735   H
2026-10-18      Brighton Crystal Palace 1642.330732 1563.551805      78.778927   H
2026-10-18         Leeds     Man United 1569.472410 1652.405876     -82.933466   D
2026-10-18 Nott'm Forest        Arsenal 1558.423477 1804.456346    -246.032869   A
2026-10-19     Tottenham       Coventry 1505.772712 1500.447537       5.325175   H


In [35]:

elo_features = all_features + ["EloDifference"]

df = df.sort_values("Date").reset_index(drop=True)

split_index = int(len(df) * 0.80)

X_train_elo = df[elo_features].iloc[:split_index]
X_test_elo = df[elo_features].iloc[split_index:]

y_train_elo = df["FTR"].iloc[:split_index]
y_test_elo = df["FTR"].iloc[split_index:]

elo_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

elo_model.fit(X_train_elo, y_train_elo)

elo_predictions = elo_model.predict(X_test_elo)

print("Elo model accuracy:",
      round(accuracy_score(y_test_elo, elo_predictions), 4))

print("Elo model macro F1:",
      round(f1_score(
          y_test_elo,
          elo_predictions,
          labels=["H", "D", "A"],
          average="macro",
          zero_division=0
      ), 4))

print(classification_report(
    y_test_elo,
    elo_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))


Elo model accuracy: 0.5274
Elo model macro F1: 0.3886
              precision    recall  f1-score   support

    Home Win       0.54      0.81      0.65       677
        Draw       0.00      0.00      0.00       369
    Away Win       0.50      0.53      0.52       488

    accuracy                           0.53      1534
   macro avg       0.35      0.45      0.39      1534
weighted avg       0.40      0.53      0.45      1534



In [36]:

# Create additional features from the Elo difference.
df["AbsEloDifference"] = df["EloDifference"].abs()
df["EloDifferenceSquared"] = df["EloDifference"] ** 2

new_elo_features = elo_features + [
    "AbsEloDifference",
    "EloDifferenceSquared"
]

# Keep the same chronological 80/20 split.
df = df.sort_values("Date").reset_index(drop=True)
split_index = int(len(df) * 0.80)

X_train_close = df[new_elo_features].iloc[:split_index]
X_test_close = df[new_elo_features].iloc[split_index:]

y_train_close = df["FTR"].iloc[:split_index]
y_test_close = df["FTR"].iloc[split_index:]

# Train both versions so we can compare them fairly.
close_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

close_model.fit(X_train_close, y_train_close)
close_predictions = close_model.predict(X_test_close)

balanced_close_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

balanced_close_model.fit(X_train_close, y_train_close)
balanced_close_predictions = balanced_close_model.predict(X_test_close)

from sklearn.metrics import accuracy_score, f1_score, classification_report

for name, predictions in [
    ("Elo + closeness features", close_predictions),
    ("Balanced Elo + closeness features", balanced_close_predictions)
]:
    print("\n", name)
    print("Accuracy:", round(
        accuracy_score(y_test_close, predictions), 4
    ))
    print("Macro F1:", round(
        f1_score(
            y_test_close,
            predictions,
            labels=["H", "D", "A"],
            average="macro",
            zero_division=0
        ), 4
    ))
    print(classification_report(
        y_test_close,
        predictions,
        labels=["H", "D", "A"],
        target_names=["Home Win", "Draw", "Away Win"],
        zero_division=0
    ))



 Elo + closeness features
Accuracy: 0.5306
Macro F1: 0.3937
              precision    recall  f1-score   support

    Home Win       0.54      0.82      0.65       677
        Draw       0.29      0.01      0.01       369
    Away Win       0.51      0.52      0.52       488

    accuracy                           0.53      1534
   macro avg       0.45      0.45      0.39      1534
weighted avg       0.47      0.53      0.46      1534


 Balanced Elo + closeness features
Accuracy: 0.4772
Macro F1: 0.455
              precision    recall  f1-score   support

    Home Win       0.62      0.53      0.57       677
        Draw       0.24      0.29      0.26       369
    Away Win       0.51      0.55      0.53       488

    accuracy                           0.48      1534
   macro avg       0.46      0.45      0.46      1534
weighted avg       0.50      0.48      0.48      1534



In [37]:

from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, log_loss
)
from scipy.stats import poisson
import numpy as np
import pandas as pd

# Predict home goals.
home_goal_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PoissonRegressor(alpha=0.1, max_iter=1000)
)

home_goal_model.fit(
    X_train_close,
    df["FTHG"].iloc[:split_index]
)

# Predict away goals.
away_goal_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PoissonRegressor(alpha=0.1, max_iter=1000)
)

away_goal_model.fit(
    X_train_close,
    df["FTAG"].iloc[:split_index]
)

# Estimate expected goals for test matches.
expected_home_goals = np.clip(
    home_goal_model.predict(X_test_close), 0.01, None
)
expected_away_goals = np.clip(
    away_goal_model.predict(X_test_close), 0.01, None
)

print("Goal models trained successfully.")
print("Example expected home goals:",
      np.round(expected_home_goals[:5], 2))
print("Example expected away goals:",
      np.round(expected_away_goals[:5], 2))


Goal models trained successfully.
Example expected home goals: [1.64 1.42 1.64 1.29 1.44]
Example expected away goals: [1.04 1.05 1.07 1.3  1.06]


In [38]:

max_goals = 6
goal_values = np.arange(max_goals + 1)

# Each row contains probabilities for scoring 0, 1, ..., 6 goals.
home_goal_probs = poisson.pmf(
    goal_values[None, :],
    expected_home_goals[:, None]
)

away_goal_probs = poisson.pmf(
    goal_values[None, :],
    expected_away_goals[:, None]
)

# Matrix dimensions: matches x home goals x away goals.
scoreline_probs = (
    home_goal_probs[:, :, None] *
    away_goal_probs[:, None, :]
)

# Combine scorelines into outcome probabilities.
home_win_probs = np.triu(
    scoreline_probs, k=1
).sum(axis=(1, 2))

draw_probs = np.einsum(
    "nii->n", scoreline_probs
)

away_win_probs = np.tril(
    scoreline_probs, k=-1
).sum(axis=(1, 2))

# Normalize to account for the tiny probability of either
# team scoring more than six goals.
outcome_probs = np.column_stack([
    home_win_probs,
    draw_probs,
    away_win_probs
])

outcome_probs = outcome_probs / outcome_probs.sum(axis=1, keepdims=True)

outcome_labels = np.array(["H", "D", "A"])
goal_model_predictions = outcome_labels[
    np.argmax(outcome_probs, axis=1)
]

print("Goal-model accuracy:",
      round(accuracy_score(y_test_close, goal_model_predictions), 4))

print("Goal-model macro F1:",
      round(f1_score(
          y_test_close,
          goal_model_predictions,
          labels=["H", "D", "A"],
          average="macro",
          zero_division=0
      ), 4))

print("Goal-model log loss:",
      round(log_loss(
          y_test_close,
          outcome_probs,
          labels=["H", "D", "A"]
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_test_close,
    goal_model_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))


Goal-model accuracy: 0.2249
Goal-model macro F1: 0.1669
Goal-model log loss: 0.9819

Classification report:
              precision    recall  f1-score   support

    Home Win       0.24      0.18      0.20       677
        Draw       0.00      0.00      0.00       369
    Away Win       0.22      0.46      0.30       488

    accuracy                           0.22      1534
   macro avg       0.15      0.21      0.17      1534
weighted avg       0.17      0.22      0.18      1534



In [39]:

scoreline_examples = []

for i in range(min(10, len(X_test_close))):
    best_score_index = np.unravel_index(
        np.argmax(scoreline_probs[i]),
        scoreline_probs[i].shape
    )

    home_goals, away_goals = best_score_index

    scoreline_examples.append({
        "Home Team": df["HomeTeam"].iloc[split_index + i],
        "Away Team": df["AwayTeam"].iloc[split_index + i],
        "Actual Result": y_test_close.iloc[i],
        "Expected Home Goals": round(expected_home_goals[i], 2),
        "Expected Away Goals": round(expected_away_goals[i], 2),
        "Likely Score": f"{home_goals}-{away_goals}",
        "Home Win %": round(outcome_probs[i, 0] * 100, 1),
        "Draw %": round(outcome_probs[i, 1] * 100, 1),
        "Away Win %": round(outcome_probs[i, 2] * 100, 1)
    })

print(pd.DataFrame(scoreline_examples).to_string(index=False))


    Home Team      Away Team Actual Result  Expected Home Goals  Expected Away Goals Likely Score  Home Win %  Draw %  Away Win %
    Newcastle Crystal Palace             D                 1.64                 1.04          1-1        24.1    24.6        51.3
       Wolves    Southampton             H                 1.42                 1.05          1-1        27.8    26.7        45.5
     Brighton      Leicester             H                 1.64                 1.07          1-1        24.7    24.6        50.7
   Man United        Arsenal             H                 1.29                 1.30          1-1        37.0    26.5        36.5
Nott'm Forest         Fulham             A                 1.44                 1.06          1-1        27.8    26.4        45.8
  Aston Villa    Southampton             H                 1.49                 1.02          1-1        25.9    26.1        48.0
       Wolves       Man City             A                 0.75                 2.30      

In [40]:

# Combine scorelines into outcome probabilities.
# Axis 1 = home goals; Axis 2 = away goals.
# Home win: home goals > away goals (lower triangle).
home_win_probs = np.tril(
    scoreline_probs, k=-1
).sum(axis=(1, 2))

# Draw: home goals == away goals (diagonal).
draw_probs = np.einsum(
    "nii->n", scoreline_probs
)

# Away win: away goals > home goals (upper triangle).
away_win_probs = np.triu(
    scoreline_probs, k=1
).sum(axis=(1, 2))

outcome_probs = np.column_stack([
    home_win_probs,
    draw_probs,
    away_win_probs
])

# Normalize for the small probability of 7+ goals.
outcome_probs = outcome_probs / outcome_probs.sum(
    axis=1, keepdims=True
)

outcome_labels = np.array(["H", "D", "A"])
goal_model_predictions = outcome_labels[
    np.argmax(outcome_probs, axis=1)
]

print("Goal-model accuracy:",
      round(accuracy_score(y_test_close, goal_model_predictions), 4))

print("Goal-model macro F1:",
      round(f1_score(
          y_test_close,
          goal_model_predictions,
          labels=["H", "D", "A"],
          average="macro",
          zero_division=0
      ), 4))

print("Goal-model log loss:",
      round(log_loss(
          y_test_close,
          outcome_probs,
          labels=["H", "D", "A"]
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_test_close,
    goal_model_predictions,
    labels=["H", "D", "A"],
    target_names=["Home Win", "Draw", "Away Win"],
    zero_division=0
))


Goal-model accuracy: 0.5346
Goal-model macro F1: 0.3942
Goal-model log loss: 1.3828

Classification report:
              precision    recall  f1-score   support

    Home Win       0.55      0.82      0.66       677
        Draw       0.00      0.00      0.00       369
    Away Win       0.51      0.54      0.53       488

    accuracy                           0.53      1534
   macro avg       0.35      0.45      0.39      1534
weighted avg       0.40      0.53      0.46      1534



In [41]:

print("Average home-win probability:",
      round(outcome_probs[:, 0].mean(), 3))

print("Average draw probability:",
      round(outcome_probs[:, 1].mean(), 3))

print("Average away-win probability:",
      round(outcome_probs[:, 2].mean(), 3))

print("Average total probability:",
      round(outcome_probs.sum(axis=1).mean(), 6))

print("\nActual outcome proportions:")
print(y_test_close.value_counts(normalize=True).reindex(["H", "D", "A"]).round(3))


Average home-win probability: 0.452
Average draw probability: 0.233
Average away-win probability: 0.315
Average total probability: 1.0

Actual outcome proportions:
FTR
H    0.441
D    0.241
A    0.318
Name: proportion, dtype: float64


In [42]:

import pandas as pd
import numpy as np

actual_one_hot = pd.DataFrame({
    "Home Win": (y_test_close == "H").astype(int).to_numpy(),
    "Draw": (y_test_close == "D").astype(int).to_numpy(),
    "Away Win": (y_test_close == "A").astype(int).to_numpy()
})

probability_names = ["Home Win", "Draw", "Away Win"]

for i, name in enumerate(probability_names):
    calibration_data = pd.DataFrame({
        "Predicted Probability": outcome_probs[:, i],
        "Actual Outcome": actual_one_hot[name].to_numpy()
    })

    calibration_data["Probability Group"] = pd.cut(
        calibration_data["Predicted Probability"],
        bins=np.linspace(0, 1, 6),
        include_lowest=True
    )

    summary = calibration_data.groupby(
        "Probability Group",
        observed=True
    ).agg(
        Matches=("Actual Outcome", "size"),
        Mean_Predicted=("Predicted Probability", "mean"),
        Actual_Frequency=("Actual Outcome", "mean")
    )

    print("\n", name)
    print(summary.round(3))




 Home Win
                   Matches  Mean_Predicted  Actual_Frequency
Probability Group                                           
(-0.001, 0.2]          114           0.148             0.167
(0.2, 0.4]             487           0.308             0.292
(0.4, 0.6]             607           0.494             0.486
(0.6, 0.8]             288           0.675             0.646
(0.8, 1.0]              38           0.841             0.921

 Draw
                   Matches  Mean_Predicted  Actual_Frequency
Probability Group                                           
(-0.001, 0.2]          252           0.165             0.179
(0.2, 0.4]            1282           0.247             0.253

 Away Win
                   Matches  Mean_Predicted  Actual_Frequency
Probability Group                                           
(-0.001, 0.2]          402           0.136             0.127
(0.2, 0.4]             718           0.293             0.302
(0.4, 0.6]             326           0.487             0

In [43]:

from sklearn.metrics import accuracy_score, f1_score, log_loss
import pandas as pd
import numpy as np

# Probability predictions from the two classification models
lr_probs = close_model.predict_proba(X_test_close)
balanced_probs = balanced_close_model.predict_proba(X_test_close)

# Ensure probability columns follow H, D, A order
def reorder_probabilities(model, X):
    raw = model.predict_proba(X)
    classes = list(model.classes_)
    return raw[:, [classes.index(c) for c in ["H", "D", "A"]]]

lr_probs = reorder_probabilities(close_model, X_test_close)
balanced_probs = reorder_probabilities(
    balanced_close_model, X_test_close
)

# Goal-based probabilities already calculated
goal_probs = outcome_probs

# Compare all models on the same test outcomes
models = {
    "Regular Logistic Regression": lr_probs,
    "Balanced Logistic Regression": balanced_probs,
    "Goal-based Model": goal_probs
}

results = []

for name, probs in models.items():
    predictions = np.array(["H", "D", "A"])[np.argmax(probs, axis=1)]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_close, predictions),
        "Macro F1": f1_score(
            y_test_close, predictions,
            labels=["H", "D", "A"],
            average="macro",
            zero_division=0
        ),
        "Log Loss": log_loss(
            y_test_close, probs, labels=["H", "D", "A"]
        )
    })

print(pd.DataFrame(results).round(4).to_string(index=False))


                       Model  Accuracy  Macro F1  Log Loss
 Regular Logistic Regression    0.5306    0.3937    1.4365
Balanced Logistic Regression    0.4772    0.4550    1.4098
            Goal-based Model    0.5346    0.3942    1.3828


In [44]:

from sklearn.metrics import log_loss
import numpy as np

print("Number of test matches:", len(y_test_close))
print("Number of probability rows:", len(outcome_probs))
print("Probability columns:", ["H", "D", "A"])

print("\nFirst 5 actual outcomes:")
print(y_test_close.iloc[:5].to_list())

print("\nFirst 5 predicted probability rows:")
print(np.round(outcome_probs[:5], 3))

print("\nLog loss checks:")
print("Goal-based:", round(
    log_loss(y_test_close, outcome_probs, labels=["H", "D", "A"]), 4
))

uniform_probs = np.full((len(y_test_close), 3), 1/3)
print("Uniform baseline:", round(
    log_loss(y_test_close, uniform_probs, labels=["H", "D", "A"]), 4
))


Number of test matches: 1534
Number of probability rows: 1534
Probability columns: ['H', 'D', 'A']

First 5 actual outcomes:
['D', 'H', 'H', 'H', 'A']

First 5 predicted probability rows:
[[0.513 0.246 0.241]
 [0.455 0.267 0.278]
 [0.507 0.246 0.247]
 [0.365 0.265 0.37 ]
 [0.458 0.264 0.278]]

Log loss checks:
Goal-based: 1.3828
Uniform baseline: 1.0986


In [45]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

actual_home_goals = df["FTHG"].iloc[split_index:].to_numpy()
actual_away_goals = df["FTAG"].iloc[split_index:].to_numpy()

print("HOME GOALS")
print("MAE:", round(mean_absolute_error(
    actual_home_goals, expected_home_goals
), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(
    actual_home_goals, expected_home_goals
)), 3))

print("\nAWAY GOALS")
print("MAE:", round(mean_absolute_error(
    actual_away_goals, expected_away_goals
), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(
    actual_away_goals, expected_away_goals
)), 3))

print("\nAverage goals")
print("Actual home goals:", round(actual_home_goals.mean(), 3))
print("Predicted home goals:", round(expected_home_goals.mean(), 3))
print("Actual away goals:", round(actual_away_goals.mean(), 3))
print("Predicted away goals:", round(expected_away_goals.mean(), 3))


HOME GOALS
MAE: 0.98
RMSE: 1.235

AWAY GOALS
MAE: 0.883
RMSE: 1.144

Average goals
Actual home goals: 1.611
Predicted home goals: 1.549
Actual away goals: 1.342
Predicted away goals: 1.214


In [46]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Training-set average goals
train_home_mean = df["FTHG"].iloc[:split_index].mean()
train_away_mean = df["FTAG"].iloc[:split_index].mean()

# Predict the same average for every test match
baseline_home = np.full(len(y_test_close), train_home_mean)
baseline_away = np.full(len(y_test_close), train_away_mean)

print("BASELINE VS MODEL")
print("Home MAE - baseline:",
      round(mean_absolute_error(actual_home_goals, baseline_home), 3))
print("Home MAE - model:",
      round(mean_absolute_error(actual_home_goals, expected_home_goals), 3))

print("\nAway MAE - baseline:",
      round(mean_absolute_error(actual_away_goals, baseline_away), 3))
print("Away MAE - model:",
      round(mean_absolute_error(actual_away_goals, expected_away_goals), 3))


BASELINE VS MODEL
Home MAE - baseline: 1.068
Home MAE - model: 0.98

Away MAE - baseline: 0.921
Away MAE - model: 0.883


In [47]:

import joblib
import os

# Keep only the historical data needed by the web app
history = df[
    ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
].copy()

# Package the models and supporting data
predictor_bundle = {
    "outcome_model": balanced_close_model,
    "home_goal_model": home_goal_model,
    "away_goal_model": away_goal_model,
    "history": history,
    "elo_ratings": ratings,
    "feature_columns": new_elo_features,
}

# Save one file for deployment
output_path = "/kaggle/working/football_predictor.pkl"

joblib.dump(predictor_bundle, output_path)

print("Export successful!")
print("File:", output_path)
print("Historical matches:", len(history))
print("Teams:", len(set(history["HomeTeam"]) | set(history["AwayTeam"])))
print("File size (MB):", round(os.path.getsize(output_path) / (1024**2), 2))


Export successful!
File: /kaggle/working/football_predictor.pkl
Historical matches: 7670
Teams: 45
File size (MB): 0.29
